## データ生成

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 5000

# ---- 真のパラメータ(これを推定で当てられるか検証する)----
beta_const = 1.0
beta_edu   = 0.8    # ← 教育の真の因果効果。これを推定で回収したい
beta_exp   = 0.3    # 経験の効果(外生)

# ---- 未観測の交絡(能力)----
ability = rng.normal(0, 1, n)          # データには入れない=未観測

# ---- 外生変数 ----
experience = rng.normal(0, 1, n)       # 経験(誤差と無相関)

# ---- 操作変数:大学近接(教育には効くが、賃金に直接は効かない)----
proximity = rng.normal(0, 1, n)        # 外部操作変数

# ---- 内生変数:教育 ----
# 教育は「操作変数 + 能力 + ノイズ」で決まる。能力経由で誤差と相関する=内生
education = (0.5 * proximity           # 関連性:操作変数が効く(第1段階の係数)
             + 0.6 * ability           # ここが内生性の源(能力と相関)
             + 0.4 * experience
             + rng.normal(0, 1, n))

# ---- 賃金:誤差に ability が混入(ここが内生性の正体)----
epsilon = 0.7 * ability + rng.normal(0, 1, n)   # 誤差が ability を含む
wage = (beta_const
        + beta_edu * education
        + beta_exp * experience
        + epsilon)                      # proximity は「直接は」入っていない=除外制約OK

df = pd.DataFrame({
    "wage": wage,
    "education": education,     # 内生
    "experience": experience,   # 外生
    "proximity": proximity,     # 操作変数
    "const": 1.0,
})
# ability は意図的に df に入れない(現実では観測できないため)


教育年数が賃金にどのような影響を与えるかに興味がある。そこで経験年数を外生変数とし、最初に次のような回帰モデルを考えた。$$y_i = \beta_{const} + x_{1i}'\beta_{edu} + x'_{2i}\beta_{exp} + \epsilon_i, \quad i = 1,...,n$$  
ただし、教育年数が誤差項と相関しているかどうか（$E[education\times \epsilon] = 0$）はわからない。教育から賃金への真の効果は $\beta_{edu} = 0.8$ であるが、このモデルはこの効果を正しく捉えることができるだろうか。

(ⅰ) OLS による推定

In [4]:
import statsmodels.api as sm

ols = sm.OLS(df["wage"], df[["const", "education", "experience"]]).fit(
    cov_type="HC1"
)
print(ols.summary())
edu_ols = ols.params["education"]
print(f"OLS推定値: {edu_ols:.3f} (真の値: {beta_edu})")

                            OLS Regression Results                            
Dep. Variable:                   wage   R-squared:                       0.615
Model:                            OLS   Adj. R-squared:                  0.615
Method:                 Least Squares   F-statistic:                     3927.
Date:                Sat, 31 Oct 2026   Prob (F-statistic):               0.00
Time:                        12:11:04   Log-Likelihood:                -7862.4
No. Observations:                5000   AIC:                         1.573e+04
Df Residuals:                    4997   BIC:                         1.575e+04
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9993      0.016     60.632      0.0

$\hat{\beta}_{edu,ols} = 1.0465$ と推定された。分析者は OLS 推定量が一致推定量であるための仮定が満たされているかがわからない。そこで内生性バイアス（説明変数と誤差項が相関）が存在していると仮定し、その原因について考える。なお $experience$ については外生変数としているので内生性は存在しないとしている。  
  
内生性バイアスは次の3つが原因であると考えられている。  
- 欠落変数：生得的な能力・意欲は、教育年数を延ばすとともに教育とは独立に賃金も上げる。しかし、能力はデータにはない。これが誤差項 $\epsilon$ に入り、教育年数と相関する。
- 同時方程式：賃金と教育が同時決定の関係にある。
- 測定誤差：教育年数の自己申告には誤差があり、推定量は0方向へバイアスが生じる。  

この中で能力の欠落がもっとも疑わしいと考えた。仮に欠落変数の発生が正しいとするならば、$$\hat{\beta}_{edu,ols} = \beta_{edu} + \underbrace{\frac{Cov(education,ability)}{Var(education)}}_{>0}\cdot \underbrace{(\text{ability の賃金への効果})}_{>0}$$つまり OLS 推定量は真の教育効果を上方にバイアスすると予測できる。しかしここで、能力のデータはないのでモデルに含めることはできないことがわかる。  
  
そこで操作変数法 (Ⅳ法) というアプローチを検討する。  
操作変数を選びたいわけだが、適切な操作変数とは本来、推計したいモデルの誤差項とは無相関であり、しかもそのモデルで用いられている内生変数と強い相関があるような変数でなければならない。賃金の決定メカニズムの外の変数で、かつシステム内の内生変数すなわち教育年数と関係があるものを探せばよい。  
そのような変数の候補として、データに大学に近接しているかという $proximity$ があることに気づく。ただし、これが本当に操作変数として適切かどうか調べる必要がある。

In [15]:
import networkx as nx

# 因果仮定(=データ外から持ち込む主張)を矢印で書く
edges = [
    ("ability", "education"),    # 交絡経路1:能力→教育
    ("ability", "wage"),         # 交絡経路2:能力→賃金
    ("experience", "education"), # 外生変数→教育
    ("experience", "wage"),      # 外生変数→賃金
    ("proximity", "education"),  # 関連性:操作変数→内生変数
    ("education", "wage"),       # ★推定したい因果効果
    # proximity → wage の直接矢印が「ない」ことが除外制約の仮定
    # proximity → ability の矢印が「ない」ことが独立性の仮定
]
G = nx.DiGraph(edges)

observed   = ["wage", "education", "experience", "proximity"]
unobserved = ["ability"]

# 目視確認:エッジ一覧と、未観測ノードの位置
print("Edges:")
for u, v in G.edges():
    tag = "(unobserved)" if u in unobserved or v in unobserved else ""
    print(f"  {u} -> {v} {tag}")

Edges:
  ability -> education (unobserved)
  ability -> wage (unobserved)
  education -> wage 
  experience -> education 
  experience -> wage 
  proximity -> education 


In [5]:
import networkx as nx

# 分析者が仮定として置くDAG(真の構造は知らないが、理論から推測して描く)
edges = [
    ("ability", "education"), ("ability", "wage"),   # 疑っている交絡(ability は未観測)
    ("experience", "education"), ("experience", "wage"),
    ("proximity", "education"),                       # 関連性(これはデータで確認可能)
    ("education", "wage"),
]
G = nx.DiGraph(edges)
unobserved = ["ability"]

def check_instrument(G, instrument, treatment, outcome, unobserved):
    relevance = nx.has_path(G, instrument, treatment)
    G_cut = G.copy(); G_cut.remove_node(treatment)
    exclusion_ok = not nx.has_path(G_cut, instrument, outcome)
    und = G.to_undirected()
    independence = all(not nx.has_path(und, instrument, u) for u in unobserved)
    return {"relevance": relevance, "exclusion": exclusion_ok,
            "independence": independence,
            "valid": relevance and exclusion_ok and independence}

print(check_instrument(G, "proximity", "education", "wage", unobserved))

{'relevance': True, 'exclusion': True, 'independence': False, 'valid': False}


In [8]:
first = sm.OLS(df["education"],
               df[["const", "proximity", "experience"]]).fit(cov_type="HC1")
print(first.summary())
f_test = first.f_test("proximity = 0")
print("第1段階 F:", float(np.ravel(f_test.fvalue)[0]))
# F > 10 が最低ライン。下回れば弱操作変数で、ここで立ち止まる

                            OLS Regression Results                            
Dep. Variable:              education   R-squared:                       0.228
Model:                            OLS   Adj. R-squared:                  0.228
Method:                 Least Squares   F-statistic:                     721.9
Date:                Sat, 31 Oct 2026   Prob (F-statistic):          3.89e-276
Time:                        13:31:52   Log-Likelihood:                -7871.3
No. Observations:                5000   AIC:                         1.575e+04
Df Residuals:                    4997   BIC:                         1.577e+04
Df Model:                           2                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0375      0.017      2.272      0.0

In [9]:
from linearmodels.iv import IV2SLS

res_2sls = IV2SLS(
    dependent=df["wage"], exog=df[["const", "experience"]],
    endog=df[["education"]], instruments=df[["proximity"]],
).fit(cov_type="robust")
print(res_2sls.summary)

edu_iv = res_2sls.params["education"]
print(f"OLS = {edu_ols:.3f},  2SLS = {edu_iv:.3f},  差 = {edu_ols - edu_iv:.3f}")
# Step 3 の予測どおり OLS > 2SLS(IVで下がる)なら、上方バイアス仮説と整合

                          IV-2SLS Estimation Summary                          
Dep. Variable:                   wage   R-squared:                      0.5713
Estimator:                    IV-2SLS   Adj. R-squared:                 0.5711
No. Observations:                5000   F-statistic:                    1748.9
Date:                Sat, Oct 31 2026   P-value (F-stat)                0.0000
Time:                        13:49:43   Distribution:                  chi2(2)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
const          1.0095     0.0174     58.145     0.0000      0.9755      1.0436
experience     0.3407     0.0220     15.464     0.00

In [10]:
print(res_2sls.wu_hausman())
# 帰無仮説「教育は外生(OLSで十分)」。有意に棄却 → 内生性あり → IVを採用すべき

Wu-Hausman test of exogeneity
H0: All endogenous variables are exogenous
Statistic: 108.0914
P-value: 0.0000
Distributed: F(1,4996)


In [14]:
summary = pd.DataFrame({
    "method": ["OLS", "2SLS(IV)"],
    "education_coef": [edu_ols, edu_iv],
    "std_error": [ols.bse["education"], res_2sls.std_errors["education"]],
})
print(summary)
print("第1段階F:", float(np.ravel(f_test.fvalue)[0]))
print("Wu-Hausman p値:", float(res_2sls.wu_hausman().pval))

     method  education_coef  std_error
0       OLS        1.046492   0.013095
1  2SLS(IV)        0.738760   0.034845
第1段階F: 931.5407768730014
Wu-Hausman p値: 4.616403982464506e-25
